# 🎓 Análisis Exploratorio de Datos — Oposiciones IT en municipios (CIDO DIBA)

**Objetivo**: responder 3 preguntas de negocio sobre los procesos selectivos de informática convocados por entidades públicas a menos de 45 km de Vic (epicentro configurable):

1. **¿Cuánto tarda en resolverse un proceso IT municipal?** (duración por grupo y sistema)
2. **¿Dónde están los puestos IT per cápita?** (normalización por población: urbano vs rural)
3. **¿Qué perfil IT piden los ayuntamientos y cuándo?** (grupos de titulación + estacionalidad)

**Fuente**: CIDO (Diputació de Barcelona) — procesos con materia *Informàtica. Noves tecnologies*. Datos enriquecidos con población municipal (Wikidata). La ingesta con scraping resumible, la limpieza con TDD y el enriquecimiento viven en `packages/` (monorepo ADR-0002); este notebook **narrativa y gráficos**, no lógica reutilizable.

## 🛠️ Paso 0: Configuración inicial

**¿Por qué?** Un solo estilo compartido para todos los gráficos (títulos, ejes y leyenda legibles), y las librerías del proyecto cargadas una vez.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
PALETA = ["#2563eb", "#dc2626", "#16a34a", "#f59e0b", "#7c3aed"]

## 📥 Paso 1: Cargar los datos

**¿De dónde salen?** `task download` scrapea el CIDO → `task clean` normaliza → `task enrich` añade población. Resultado: `data/processed/oposicions_enriquit.csv` (1 fila = 1 hito del proceso, con los datos del proceso desnormalizados).

**Nota RGPD (ADR-0004)**: este dataset solo contiene metadatos de procesos — jamás datos de aspirantes.

In [ ]:
from pathlib import Path

RUTA_DATOS = Path("data/processed/oposicions_enriquit.csv")
if not RUTA_DATOS.exists():          # nbconvert ejecuta desde notebooks/
    RUTA_DATOS = Path("..") / RUTA_DATOS
df = pd.read_csv(RUTA_DATOS, dtype={"proceso_id": str})

print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]} | Procesos únicos: {df['proceso_id'].nunique()}")

## 👀 Paso 2: Primera exploración

**¿Por qué?** Ver cómo lucen los datos, qué columnas tenemos y de qué tipos, antes de graficar nada.

In [ ]:
df.head(3)

In [ ]:
print(df.columns.tolist())
df.info()

## 📊 Paso 3: Estadística descriptiva

**¿Por qué?** `describe()` resume las numéricas (count, media, percentiles) y `describe(exclude='number')` las categóricas (cuántas únicas, la más frecuente). Sirve para detectar valores extraños antes de graficar.

In [ ]:
df.describe()

In [ ]:
df.describe(exclude='number')

## 🧹 Paso 4: Valores faltantes (ya gestionados en el pipeline)

La limpieza NO ocurre aquí: vive en `eda_core` con **TDD** (test primero, ADR-0005 con los catálogos y la regla del `None` honesto). Este notebook solo la **audita**: cuántos vacíos quedan por columna y por qué son honestos.

**¿Por qué no imputamos?** Imputar estados o plazas inventaría datos que el portal no publicó; el CEO tomaría decisiones sobre ficción (ADR-0005).

In [ ]:
nulos = (df == "").sum()
pct = (nulos / len(df) * 100).round(1)
print(pd.DataFrame({"vacios": nulos, "%": pct})[nulos > 0].sort_values("%", ascending=False))

## 📈 Paso 5: Análisis univariado (≥3 variables)

**¿Qué es?** Una variable a la vez, para entender su distribución antes de cruzarla con nada.

### 5.1 Duración del proceso (meses) — solo las resueltas

**¿Por qué solo resueltas?** Un proceso abierto no tiene duración: su `durada_mesos` es vacío por diseño honesto (ADR-0005), no por error.

In [ ]:
duracion = pd.to_numeric(df["durada_mesos"], errors="coerce").dropna()

plt.hist(duracion, bins=30, color=PALETA[0], edgecolor="white")
plt.title("Distribución de la duración de los procesos (meses)")
plt.xlabel("Meses entre convocatoria y resolución")
plt.ylabel("Nº de hitos")
plt.show()

print(f"Procesos con duración computable: {duracion.notna().sum()}")
print(duracion.describe().round(1))

### 5.2 Estado del proceso (categórica)

**¿Por qué?** Cuántos procesos siguen vivos (abiertas) frente al histórico resuelto y cerrado.

In [ ]:
orden = ["abierta", "pendiente", "cerrada", "resuelta", "no vigente"]
conteo = df["estado"].fillna("sin estado").value_counts()

conteo.reindex([e for e in orden if e in conteo.index] + ["sin estado"]).plot(
    kind="bar", color=PALETA[1], edgecolor="white")
plt.title("Estado de los procesos (catálogo normalizado)")
plt.xlabel("Estado"); plt.ylabel("Nº de hitos"); plt.xticks(rotation=30, ha="right")
plt.show()

### 5.3 Plazas publicadas

**¿Por qué?** La mayoría de procesos convoca 1 plaza; las convocatorias de varias plazas son la excepción (y las bolsas no cuantifican).

In [ ]:
plazas = pd.to_numeric(df["plazas"], errors="coerce").dropna()

plazas.value_counts().sort_index().plot(kind="bar", color=PALETA[2], edgecolor="white")
plt.title("Plazas convocadas por proceso")
plt.xlabel("Nº de plazas"); plt.ylabel("Nº de hitos"); plt.xticks(rotation=0)
plt.show()

print(f"Procesos con plaza publicada: {plazas.notna().sum()} | sin cuantificar: {(df['plazas']=='').sum()}")

## 🔗 Paso 6: Análisis bivariado (≥3 relaciones)

**¿Qué es?** Cruzar dos variables para ver si se relacionan.

### 6.1 Duración × Grupo de titulación

**¿Por qué?** ¿Los procesos de perfiles senior (A1/A2) tardan lo mismo que los de FP (C1/C2)?

In [ ]:
d = df.dropna(subset=["durada_mesos"]).copy()
d["durada_mesos"] = pd.to_numeric(d["durada_mesos"])
top_grupos = d["grupo"].value_counts().head(5).index

sns.boxplot(data=d[d["grupo"].isin(top_grupos)], x="grupo", y="durada_mesos",
            hue="grupo", legend=False, palette=PALETA)
plt.title("Duración del proceso por grupo de titulación")
plt.xlabel("Grupo"); plt.ylabel("Meses hasta resolución")
plt.show()

### 6.2 Plazas × Población del municipio (escala log)

**¿Por qué log?** Poblaciones de 120 a 225.000 hab — sin log, todo se aplasta contra los grandes.

In [ ]:
p = df[(df["plazas"] != "") & (df["poblacion_municipi"] != "")].copy()
p["plazas"] = pd.to_numeric(p["plazas"])
p["poblacion_municipi"] = pd.to_numeric(p["poblacion_municipi"])
p = p.drop_duplicates(subset="proceso_id")

plt.scatter(p["poblacion_municipi"], p["plazas"], alpha=0.4, color=PALETA[3])
plt.xscale("log"); plt.yscale("log")
plt.title("Plazas convocadas vs población del municipio")
plt.xlabel("Población del municipio (log)"); plt.ylabel("Plazas (log)")
plt.show()

### 6.3 Duración × Tamaño del municipio (urbano / intermedio / rural)

**¿Por qué?** La pregunta 2 en su forma bivariada: ¿los municipios pequeños tardan más en resolver?

In [ ]:
sns.boxplot(data=d, x="urbanizacion", y="durada_mesos",
            hue="urbanizacion", legend=False, order=["rural", "intermedio", "urbano"],
            palette=["#16a34a", "#f59e0b", "#2563eb"])
plt.title("Duración del proceso por tamaño del municipio")
plt.xlabel("Tamaño del municipio (hab: rural <5k, intermedio 5-20k, urbano ≥20k)")
plt.ylabel("Meses hasta resolución")
plt.show()

## ⚠️ Paso 7: Valores atípicos (outliers) por IQR

**¿Qué es el IQR?** El rango intercuartílico: entre el percentil 25 y el 75. Un valor es atípico si cae fuera de `[Q1 - 1.5·IQR, Q3 + 1.5·IQR]`.

**¿Por qué mirarlos?** Procesos de 5+ años suelen ser "atascados" (impugnaciones, bolsas congeladas) — no errores: son hallazgos.

In [ ]:
q1, q3 = duracion.quantile([0.25, 0.75])
iqr = q3 - q1
limite = q3 + 1.5 * iqr
atipicos = duracion[duracion > limite]

plt.boxplot(duracion, vert=False)
plt.axvline(limite, color=PALETA[1], linestyle="--", label=f"límite IQR = {limite:.0f} meses")
plt.title("Duración de los procesos y outliers")
plt.xlabel("Meses"); plt.yticks([]); plt.legend()
plt.show()

print(f"Outliers: {len(atipicos)} de {len(duracion)} procesos ({len(atipicos)/len(duracion)*100:.0f}%)")
print(f"El más largo: {atipicos.max():.0f} meses")

## 📊 Paso 8: Tabla dinámica (pivot_table)

**¿Por qué?** Resumir el cruce año × estado en una tabla que el informe ejecutivo pueda citar.

In [ ]:
df["anio"] = df["fecha_convocatoria"].str[:4]
pivot = df.pivot_table(index="anio", columns="estado", values="proceso_id",
                       aggfunc="nunique", fill_value=0)
pivot

## 🔥 Paso 9: Matriz de correlación

**¿Por qué Spearman y no Pearson?** Las variables (plazas, población, duración) están muy sesgadas; Spearman mide relaciones monótonas sin asumir linealidad.

**Ojo**: correlación ≠ causalidad.

In [ ]:
numericas = df.drop_duplicates(subset="proceso_id")[
    ["plazas", "durada_mesos", "poblacion_municipi", "dist_km_vic", "places_per_mil_hab"]
].apply(pd.to_numeric, errors="coerce")

corr = numericas.corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Correlación (Spearman) entre variables del proceso")
plt.show()

## 📉 Paso 10: Serie temporal con transformación justificada

**¿Por qué la media móvil de 12 meses?** Las convocatorias mensuales son muy volátiles (estacionalidad + tamaños municipales dispares); la media móvil 12m suaviza el ruido y deja ver la **tendencia** del ciclo.

**Además, escala log**: los picos de algunos años dominarían la escala lineal.

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha_convocatoria"], errors="coerce")
mensual = df.dropna(subset=["fecha"]).drop_duplicates(subset="proceso_id").set_index("fecha")
serie = mensual.resample("ME").size()

plt.plot(serie.index, serie.values, alpha=0.3, label="mensual (ruido)", color=PALETA[0])
plt.plot(serie.index, serie.rolling(12).mean(), linewidth=2.5, label="media móvil 12 meses", color=PALETA[1])
plt.yscale("log")
plt.title("Convocatorias IT por mes (escala log)")
plt.xlabel("Fecha"); plt.ylabel("Procesos convocados (log)")
plt.legend()
plt.show()

## ❓ Paso 11: Las 3 preguntas de negocio

### P1 — ¿Cuánto tarda en resolverse un proceso IT municipal?

**Hallazgo esperable**: la mediana está por debajo de la media (distribución sesgada: hay procesos atascadísimos que arrastran la media). El informe ejecutivo citará la **mediana**, no la media.

In [ ]:
resumen_p1 = d.groupby("grupo")["durada_mesos"].agg(
    n="count", mediana="median", media="mean").round(1)
print(resumen_p1)
print(f"\nMediana global: {d['durada_mesos'].median():.1f} meses | Media: {d['durada_mesos'].mean():.1f} meses")

In [ ]:
sns.histplot(d["durada_mesos"], bins=40, color=PALETA[0])
plt.axvline(d["durada_mesos"].median(), color=PALETA[1], linestyle="--",
            label=f"mediana = {d['durada_mesos'].median():.1f} meses")
plt.axvline(d["durada_mesos"].mean(), color="#111827", linestyle=":",
            label=f"media = {d['durada_mesos'].mean():.1f} meses")
plt.title("P1 — Duración convocatoria→resolución (resueltas)")
plt.xlabel("Meses"); plt.ylabel("Procesos"); plt.legend()
plt.show()

### P2 — ¿Dónde están los puestos IT per cápita?

**¿Por qué per cápita?** Sin normalizar, Barcelona-área siempre gana por tamaño. Plazas **por 1.000 habitantes** es la comparación justa (mín. 1.000 hab para no amplificar municipios minúsculos con 1 plaza).

In [ ]:
munis = (df[df["poblacion_municipi"] != ""]
         .assign(pob=lambda x: pd.to_numeric(x["poblacion_municipi"]))
         .query("pob >= 1000")
         .groupby(["municipi", "urbanizacion"])["proceso_id"].nunique()
         .rename("procesos").reset_index())
munis = munis.merge(
    df.drop_duplicates("municipi")[["municipi", "poblacion_municipi"]],
    on="municipi")
munis["pob"] = pd.to_numeric(munis["poblacion_municipi"])
munis["por_1k"] = (munis["procesos"] / (munis["pob"] / 1000)).round(2)
top = munis.nlargest(10, "por_1k")

plt.barh(top["municipi"], top["por_1k"], color=PALETA[4])
plt.title("P2 — Municipios con más procesos IT per cápita (procesos por 1.000 hab)")
plt.xlabel("Procesos por 1.000 habitantes"); plt.gca().invert_yaxis()
plt.show()
print(top[["municipi", "urbanizacion", "procesos", "por_1k"]].to_string(index=False))

### P3 — ¿Qué perfil IT piden y cuándo?

**¿Por qué?** ¿FP (C1/C2) o universitarios (A1/A2)? ¿Hay "temporada" de convocatorias? — la pregunta que decide a qué perfil formarse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.drop_duplicates("proceso_id")["grupo"].value_counts().head(6).plot(
    kind="barh", ax=axes[0], color=PALETA[0])
axes[0].set_title("Procesos por grupo de titulación")
axes[0].set_xlabel("Procesos"); axes[0].invert_yaxis()

df.dropna(subset=["fecha"]).drop_duplicates("proceso_id")["fecha"].dt.month.value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color=PALETA[3])
axes[1].set_title("Estacionalidad: convocatorias por mes")
axes[1].set_xlabel("Mes"); axes[1].set_ylabel("Procesos"); axes[1].set_xticklabels(
    ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"], rotation=0)

plt.tight_layout(); plt.show()

## 🎯 Conclusiones

*(las respuestas escritas definitivas se redactan tras ejecutar el notebook con los datos reales — este es el esqueleto del análisis)*

- **P1 (duración)**: la mediana de resolución es inferior a la media → hay una cola de procesos atascados (outliers IQR) que distorsionan la media. Para el CEO: citar mediana y el % de outliers.
- **P2 (per cápita)**: los municipios pequeños dominan el ranking per cápita — la normalización invierte el ranking bruto. Para el CEO: las oportunidades IT no están solo en los grandes.
- **P3 (perfiles y estacionalidad)**: el perfil demandado se concentra en grupos de FP/campus medio y la estacionalidad muestra meses valle evidentes — información directa para planificar formación.

**Limitaciones honestas**: duración solo computable en resueltas (~1/3 del histórico); procesos supramunicipales (Diputació, universidades) sin denominador de población; el portal no publica aspirantes (no hay "competencia por plaza" salvo en los PDFs de bolsa, fuera del alcance RGPD-actual).